# Amazon SageMaker Asynchronous Inference using the SageMaker Python SDK


---

This notebook's CI test result for us-west-2 is as follows. CI test results in other regions can be found at the end of the notebook. 

![This us-west-2 badge failed to load. Check your device's internet connectivity, otherwise the service is currently unavailable](https://prod.us-west-2.tcx-beacon.docs.aws.dev/sagemaker-nb/us-west-2/deploy_and_monitor|sm-async_inference_with_python_sdk|sm-async_inference_with_python_sdk.ipynb)

---

_**A new near real-time Inference option for generating machine learning model predictions**_

This example makes use of the [SageMaker Python SDK](https://sagemaker.readthedocs.io/en/stable/) to deploy an Asynchronous Endpoint. While the [Async-Inference-Walkthrough.ipynb](https://github.com/aws/amazon-sagemaker-examples/blob/main/async-inference/Async-Inference-Walkthrough.ipynb) example  makes use of [Boto3](https://boto3.amazonaws.com/v1/documentation/api/latest/guide/quickstart.html). 

**Table of Contents**

* [Background](#background)
* [Notebook Scope](#scope)
* [Overview and sample end to end flow](#overview)
* [Section 1 - Setup](#setup) 
    * [Create SageMaker Model](#createmodel)
    * [Create AsyncInferenceConfig](#asyncinferenceconfig)
    * [Create Endpoint](#create-endpoint)
    * [Setup AutoScaling policy (Optional)](#setup-autoscaling)
* [Section 2 - Using the Endpoint](#endpoint) 
    * [Get Prediction from the Endpoint](#invoke-endpoint)
    * [Check Output Location](#check-output)
    * [Multiple Predictions](#multiple-invoke)  
* [Section 3 - Clean up](#clean)

### Background <a id='background'></a>  
Amazon SageMaker Asynchronous Inference is a new capability in SageMaker that queues incoming requests and processes them asynchronously. SageMaker currently offers two inference options for customers to deploy machine learning models: 1) a real-time option for low-latency workloads 2) Batch transform, an offline option to process inference requests on batches of data available upfront. Real-time inference is suited for workloads with payload sizes of less than 6 MB and require inference requests to be processed within 60 seconds. Batch transform is suitable for offline inference on batches of data. 

Asynchronous inference is a new inference option for near real-time inference needs. Requests can take up to 15 minutes to process and have payload sizes of up to 1 GB. Asynchronous inference is suitable for workloads that do not have sub-second latency requirements and have relaxed latency requirements. For example, you might need to process an inference on a large image of several MBs within 5 minutes. In addition, asynchronous inference endpoints let you control costs by scaling down endpoints instance count to zero when they are idle, so you only pay when your endpoints are processing requests. 

### Notebook scope <a id='scope'></a>  
This notebook provides an introduction to the SageMaker Asynchronous inference capability. This notebook will cover the steps required to create an asynchronous inference endpoint and test it with some sample requests. 

### Overview and sample end to end flow <a id='overview'></a>
Asynchronous inference endpoints have many similarities (and some key differences) compared to real-time endpoints. The process to create asynchronous endpoints is similar to real-time endpoints. You need to create: a SageMaker Model and then an endpoint. However, there are specific configuration parameters specific to asynchronous inference endpoints which we will explore below. 

Invocation of asynchronous endpoints differ from real-time endpoints. Rather than pass request payload inline with the request, you upload the payload to Amazon S3 and pass an Amazon S3 URI as a part of the request. Upon receiving the request, SageMaker provides you with a token with the output location where the result will be placed once processed. Internally, SageMaker maintains a queue with these requests and processes them. During endpoint creation, you can optionally specify an Amazon SNS topic to receive success or error notifications. Once you receive the notification that your inference request has been successfully processed, you can access the result in the output Amazon S3 location. 

The diagram below provides a visual overview of the end-to-end flow with Asynchronous inference endpoint.

![title](images/e2e.png)

## Dataset

We're about to work with the [Titanic dataset](https://www.openml.org/d/40945)[1]. From the dataset documentation:

> The original Titanic dataset, describing the survival status of individual passengers on the Titanic. The titanic data does not contain information from the crew, but it does contain actual ages of half of the passengers. The principal source for data about Titanic passengers is the Encyclopedia Titanica. The datasets used here were begun by a variety of researchers. One of the original sources is Eaton & Haas (1994) Titanic: Triumph and Tragedy, Patrick Stephens Ltd, which includes a passenger list created by many researchers and edited by Michael A. Findlay.
>
> Thomas Cason of UVa has greatly updated and improved the Titanic data frame using the Encyclopedia Titanica and created the dataset here. Some duplicate passengers have been dropped, many errors corrected, many missing ages filled in, and new variables created.


---
## 1. Setup <a id='setup'></a>

First we ensure we have an updated version of the SageMaker Python SDK, which includes the latest SageMaker features:

Import the required python libraries:

In [1]:
# [exec-copy] install steps neutralized; .v3test-venv already has V3 SDK
print('skipping pip install in execution copy')

skipping pip install in execution copy


In [2]:
import boto3
from time import gmtime, strftime
from datetime import datetime

# V3: Session and get_execution_role live under sagemaker.core.helper.session_helper
from sagemaker.core.helper.session_helper import Session, get_execution_role

boto_session = boto3.session.Session(region_name="us-west-1")
sm_session = Session(boto_session=boto_session)
region = boto_session.region_name

[07/16/26 10:00:27] INFO     Found credentials in shared credentials file: ~/.aws/credentials   ]8;id=12565431;file:///Users/lucasjia/pysdk/pysdk-workspace/.v3test-venv/lib/python3.12/site-packages/botocore/credentials.py\credentials.py]8;;\:]8;id=12565432;file:///Users/lucasjia/pysdk/pysdk-workspace/.v3test-venv/lib/python3.12/site-packages/botocore/credentials.py#1392\1392]8;;\

sagemaker.config INFO - Not applying SDK defaults from location: /Library/Application Support/sagemaker/config.yaml


sagemaker.config INFO - Not applying SDK defaults from location: /Users/lucasjia/Library/Application Support/sagemaker/config.yaml


If you want to make use of SNS notifcations, provide your role access to SNS. Go the AWS IAM console (https://console.aws.amazon.com/iam/home) and add the following policies to your IAM Role:
    
    
    * (Optional) Amazon SNS access: Add `sns:Publish` on the topics you define. Apply this if you plan to use Amazon SNS to receive notifications.

```json
{
    "Version": "2012-10-17",
    "Statement": [
        {
            "Action": [
                "sns:Publish"
            ],
            "Effect": "Allow",
            "Resource": "arn:aws:sns:us-east-2:123456789012:MyTopic"
        }
    ]
}
```

* (Optional) KMS decrypt, encrypt if your Amazon S3 bucket is encrypted.

In [3]:
# Download the Input files and model from S3 bucket
!mkdir input
!mkdir model

s3 = boto3.client("s3")
for key in s3.list_objects(
    Bucket=f"sagemaker-example-files-prod-{region}", Prefix="models/async-inference/input-files/"
)["Contents"]:
    s3.download_file(
        f"sagemaker-example-files-prod-{region}", key["Key"], "input/" + key["Key"].split("/")[-1]
    )
s3.download_file(
    f"sagemaker-example-files-prod-{region}",
    "models/async-inference/demo-xgboost-model.tar.gz",
    "model/demo-xgboost-model.tar.gz",
)

                    INFO     Found credentials in shared credentials file: ~/.aws/credentials   ]8;id=12565437;file:///Users/lucasjia/pysdk/pysdk-workspace/.v3test-venv/lib/python3.12/site-packages/botocore/credentials.py\credentials.py]8;;\:]8;id=12565438;file:///Users/lucasjia/pysdk/pysdk-workspace/.v3test-venv/lib/python3.12/site-packages/botocore/credentials.py#1392\1392]8;;\

Specify your SageMaker IAM Role (`sm_role`) and Amazon S3 bucket (`s3_bucket`). You can optionally use a default SageMaker Session IAM Role and Amazon S3 bucket. Make sure the role you use has the necessary permissions for SageMaker, Amazon S3, and optionally Amazon SNS.

In [4]:
sm_role = get_execution_role()
# Feel free to use your own role here
# sm_role = "arn:aws:iam::123456789012:role/sagemaker-custom-role"
print(f"Using Role: {sm_role}")
s3_bucket = sm_session.default_bucket()
default_bucket_prefix = sm_session.default_bucket_prefix
print(f"Will use bucket '{s3_bucket}' for storing all resources related to this notebook")

Using Role: arn:aws:iam::729646638167:role/Admin


Will use bucket 'sagemaker-us-west-1-729646638167' for storing all resources related to this notebook


In [5]:
bucket_prefix = "DEMO-async-inference"

# If a default bucket prefix is specified, append it to the s3 path
if default_bucket_prefix:
    bucket_prefix = f"{default_bucket_prefix}/{bucket_prefix}"

resource_name = "AsyncInferenceDemo-{}-{}"

Next, you will create a model with `CreateModel`, an endpoint configuration with `CreateEndpointConfig`, and then an endpoint with the `CreateEndpoint` API.


### 1.1 Create a SageMaker Model <a id='createmodel'></a>
Specify the location of the pre-trained model stored in Amazon S3. This example uses a pre-trained XGBoost model name demo-xgboost-model.tar.gz. The full Amazon S3 URI is stored in a string variable `model_url`. 

In [6]:
model_s3_key = f"{bucket_prefix}/demo-xgboost-model.tar.gz"
model_url = f"s3://{s3_bucket}/{model_s3_key}"
print(f"Uploading Model to {model_url}")

with open("model/demo-xgboost-model.tar.gz", "rb") as model_file:
    boto_session.resource("s3").Bucket(s3_bucket).Object(model_s3_key).upload_fileobj(model_file)

Uploading Model to s3://sagemaker-us-west-1-729646638167/DEMO-async-inference/demo-xgboost-model.tar.gz


Specify a primary container. For the primary container, you specify the Docker image that contains inference code, artifacts (from prior training), and a custom environment map that the inference code uses when you deploy the model for predictions. In this example, we specify an XGBoost built-in algorithm container image.

In [7]:
# V3: image_uris moved to sagemaker.core.image_uris
from sagemaker.core import image_uris

# Specify an AWS container image and region as desired
container = image_uris.retrieve(region=region, framework="xgboost", version="0.90-1")

[07/16/26 10:00:33] INFO     Found credentials in shared credentials file: ~/.aws/credentials   ]8;id=12565443;file:///Users/lucasjia/pysdk/pysdk-workspace/.v3test-venv/lib/python3.12/site-packages/botocore/credentials.py\credentials.py]8;;\:]8;id=12565444;file:///Users/lucasjia/pysdk/pysdk-workspace/.v3test-venv/lib/python3.12/site-packages/botocore/credentials.py#1392\1392]8;;\

[07/16/26 10:00:34] INFO     Defaulting to only available Python version: py3                     ]8;id=12565451;file:///Users/lucasjia/pysdk/pysdk-workspace/sagemaker-python-sdk-lucas/sagemaker-core/src/sagemaker/core/image_uris.py\image_uris.py]8;;\:]8;id=12565452;file:///Users/lucasjia/pysdk/pysdk-workspace/sagemaker-python-sdk-lucas/sagemaker-core/src/sagemaker/core/image_uris.py#615\615]8;;\

                    INFO     Defaulting to only supported image scope: cpu.                       ]8;id=12565458;file:///Users/lucasjia/pysdk/pysdk-workspace/sagemaker-python-sdk-lucas/sagemaker-core/src/sagemaker/core/image_uris.py\image_uris.py]8;;\:]8;id=12565459;file:///Users/lucasjia/pysdk/pysdk-workspace/sagemaker-python-sdk-lucas/sagemaker-core/src/sagemaker/core/image_uris.py#539\539]8;;\

Create a SageMaker [`Model`](https://sagemaker.readthedocs.io/en/stable/api/inference/model.html#model) by specifying the `name`, the `role` (the ARN of the IAM role that Amazon SageMaker can assume to access model artifacts/ docker images for deployment), and the `image_uri` of the XGBoost built-in algorithm container image.

In [8]:
# V3: the V2 Model + Predictor pattern is replaced by ModelBuilder.
# Here we deploy a pre-trained model artifact with a first-party (built-in) XGBoost
# container image and no custom inference code, which ModelBuilder handles via its
# pass-through path (image_uri is a 1P image, model_path points at the S3 artifact).
from sagemaker.serve import ModelBuilder

model_name = resource_name.format("Model", datetime.now().strftime("%Y-%m-%d-%H-%M-%S"))

model_builder = ModelBuilder(
    image_uri=container,
    model_path=model_url,
    role_arn=sm_role,
    instance_type="ml.m5.xlarge",
    sagemaker_session=sm_session,
)
model_name

'AsyncInferenceDemo-Model-2026-07-16-10-00-36'

### 1.2 Create AsyncInferenceConfig <a id='asyncinferenceconfig'></a>

 Specify the [`AsyncInferenceConfig`](https://sagemaker.readthedocs.io/en/stable/api/inference/async_inference.html?highlight=AsyncInferenceConfig#sagemaker.async_inference.async_inference_config.AsyncInferenceConfig) object and provide an output Amazon S3 location for `output_path`. You can optionally specify Amazon SNS topics on which to send notifications about prediction results.

In [9]:
# V3: AsyncInferenceConfig moved to sagemaker.core.inference_config
from sagemaker.core.inference_config import AsyncInferenceConfig

async_config = AsyncInferenceConfig(
    output_path=f"s3://{s3_bucket}/{bucket_prefix}/output",
    max_concurrent_invocations_per_instance=4,
    # Optionally specify Amazon SNS topics
    # notification_config = {
    # "SuccessTopic": "arn:aws:sns:<aws-region>:<account-id>:<topic-name>",
    # "ErrorTopic": "arn:aws:sns:<aws-region>:<account-id>:<topic-name>",
    # }
)

### 1.3 Create Endpoint <a id='create-endpoint'></a>

Once you have your [`Model`](https://sagemaker.readthedocs.io/en/stable/api/inference/model.html#model) and [`AsyncInferenceConfig`](https://sagemaker.readthedocs.io/en/stable/api/inference/async_inference.html?highlight=AsyncInferenceConfig#sagemaker.async_inference.async_inference_config.AsyncInferenceConfig), you can use the [`.deploy()`](https://sagemaker.readthedocs.io/en/stable/api/inference/model.html#sagemaker.model.Model.deploy) API to create your endpoint. In the `.deploy()` API we set our `instance_type`, `initial_instance_count` and pass the [`async_config`](#asyncinferenceconfig) we created above. 

The endpoint name must be unique within an AWS Region in your AWS account. 

In [10]:
endpoint_name = resource_name.format("Endpoint", datetime.now().strftime("%Y-%m-%d-%H-%M-%S"))

# Build the model, then deploy an asynchronous endpoint.
# In V3, ModelBuilder.deploy() returns a sagemaker.core.resources.Endpoint
# (not the deprecated AsyncPredictor). Pass the AsyncInferenceConfig via inference_config.
model_builder.build(model_name=model_name)

async_endpoint = model_builder.deploy(
    endpoint_name=endpoint_name,
    initial_instance_count=1,
    inference_config=async_config,
)
endpoint_name

[07/16/26 10:00:36] INFO     SageMaker Python SDK will collect telemetry to help us better ]8;id=12565466;file:///Users/lucasjia/pysdk/pysdk-workspace/sagemaker-python-sdk-lucas/sagemaker-core/src/sagemaker/core/telemetry/telemetry_logging.py\telemetry_logging.py]8;;\:]8;id=12565467;file:///Users/lucasjia/pysdk/pysdk-workspace/sagemaker-python-sdk-lucas/sagemaker-core/src/sagemaker/core/telemetry/telemetry_logging.py#287\287]8;;\
                             understand our user's needs, diagnose issues, and deliver                             
                             additional features.                                                                  
                             To opt out of telemetry, please disable via TelemetryOptOut                           
                             parameter in SDK defaults config. For more information, refer                         
                             to                                                                                    
                             https://sagemaker.readthedocs.io/en/stable/overview.html#conf                         
                             iguring-and-using-defaults-with-the-sagemaker-python-sdk.                             

                    DEBUG    No ModelMetadata provided. ModelBuilder is not handling    ]8;id=12565474;file:///Users/lucasjia/pysdk/pysdk-workspace/sagemaker-python-sdk-lucas/sagemaker-serve/src/sagemaker/serve/model_builder_utils.py\model_builder_utils.py]8;;\:]8;id=12565475;file:///Users/lucasjia/pysdk/pysdk-workspace/sagemaker-python-sdk-lucas/sagemaker-serve/src/sagemaker/serve/model_builder_utils.py#1388\1388]8;;\
                             MLflow model input                                                                    

                    INFO     Role 'arn:aws:iam::729646638167:role/Admin' validated for     ]8;id=12565482;file:///Users/lucasjia/pysdk/pysdk-workspace/sagemaker-python-sdk-lucas/sagemaker-core/src/sagemaker/core/helper/iam_role_resolver.py\iam_role_resolver.py]8;;\:]8;id=12565483;file:///Users/lucasjia/pysdk/pysdk-workspace/sagemaker-python-sdk-lucas/sagemaker-core/src/sagemaker/core/helper/iam_role_resolver.py#598\598]8;;\
                             serving. Using it.                                                                    

[07/16/26 10:00:37] INFO     Role 'arn:aws:iam::729646638167:role/Admin' validated for     ]8;id=12565488;file:///Users/lucasjia/pysdk/pysdk-workspace/sagemaker-python-sdk-lucas/sagemaker-core/src/sagemaker/core/helper/iam_role_resolver.py\iam_role_resolver.py]8;;\:]8;id=12565489;file:///Users/lucasjia/pysdk/pysdk-workspace/sagemaker-python-sdk-lucas/sagemaker-core/src/sagemaker/core/helper/iam_role_resolver.py#598\598]8;;\
                             serving. Using it.                                                                    

                    INFO     Creating model with name:                                       ]8;id=12565496;file:///Users/lucasjia/pysdk/pysdk-workspace/sagemaker-python-sdk-lucas/sagemaker-core/src/sagemaker/core/helper/session_helper.py\session_helper.py]8;;\:]8;id=12565497;file:///Users/lucasjia/pysdk/pysdk-workspace/sagemaker-python-sdk-lucas/sagemaker-core/src/sagemaker/core/helper/session_helper.py#1922\1922]8;;\
                             AsyncInferenceDemo-Model-2026-07-16-10-00-36                                          

...

169.254.178.2 - - [16/Jul/2026:17:03:21 +0000] "GET /ping HTTP/1.1" 200 0 "-" "AHC/2.0"

169.254.178.2 - - [16/Jul/2026:17:03:26 +0000] "GET /ping HTTP/1.1" 200 0 "-" "AHC/2.0"

169.254.178.2 - - [16/Jul/2026:17:03:31 +0000] "GET /ping HTTP/1.1" 200 0 "-" "AHC/2.0"

[07/16/26 10:03:40] INFO     ✅ Deployment successful: Endpoint                               ]8;id=12565565;file:///Users/lucasjia/pysdk/pysdk-workspace/sagemaker-python-sdk-lucas/sagemaker-serve/src/sagemaker/serve/model_builder.py\model_builder.py]8;;\:]8;id=12565566;file:///Users/lucasjia/pysdk/pysdk-workspace/sagemaker-python-sdk-lucas/sagemaker-serve/src/sagemaker/serve/model_builder.py#2971\2971]8;;\
                             'AsyncInferenceDemo-Endpoint-2026-07-16-10-00-36' using None in                       
                             SAGEMAKER_ENDPOINT mode (ARN:                                                         
                             arn:aws:sagemaker:us-west-1:729646638167:endpoint/AsyncInference                      
                             Demo-Endpoint-2026-07-16-10-00-36)                                                    

'AsyncInferenceDemo-Endpoint-2026-07-16-10-00-36'

### 1.4 Setup AutoScaling policy (Optional)    <a id='setup-autoscaling'></a>

This section describes how to configure autoscaling on your asynchronous endpoint using Application Autoscaling. You need to first register your endpoint variant with Application Autoscaling, define a scaling policy, and then apply the scaling policy. In this configuration, we use a custom metric, `CustomizedMetricSpecification`, called `ApproximateBacklogSizePerInstance`. Please refer to the SageMaker Developer guide for a detailed list of metrics available with your asynchronous inference endpoint.

In [11]:
client = boto3.client(
    "application-autoscaling"
)  # Common class representing Application Auto Scaling for SageMaker amongst other services

resource_id = (
    "endpoint/" + endpoint_name + "/variant/" + "AllTraffic"
)  # This is the format in which application autoscaling references the endpoint

# Configure Autoscaling on asynchronous endpoint down to zero instances
response = client.register_scalable_target(
    ServiceNamespace="sagemaker",
    ResourceId=resource_id,
    ScalableDimension="sagemaker:variant:DesiredInstanceCount",
    MinCapacity=0,
    MaxCapacity=5,
)

response = client.put_scaling_policy(
    PolicyName="Invocations-ScalingPolicy",
    ServiceNamespace="sagemaker",  # The namespace of the AWS service that provides the resource.
    ResourceId=resource_id,  # Endpoint name
    ScalableDimension="sagemaker:variant:DesiredInstanceCount",  # SageMaker supports only Instance Count
    PolicyType="TargetTrackingScaling",  # 'StepScaling'|'TargetTrackingScaling'
    TargetTrackingScalingPolicyConfiguration={
        "TargetValue": 5.0,  # The target value for the metric. - here the metric is - SageMakerVariantInvocationsPerInstance
        "CustomizedMetricSpecification": {
            "MetricName": "ApproximateBacklogSizePerInstance",
            "Namespace": "AWS/SageMaker",
            "Dimensions": [{"Name": "EndpointName", "Value": endpoint_name}],
            "Statistic": "Average",
        },
        "ScaleInCooldown": 600,  # The cooldown period helps you prevent your Auto Scaling group from launching or terminating
        # additional instances before the effects of previous activities are visible.
        # You can configure the length of time based on your instance startup time or other application needs.
        # ScaleInCooldown - The amount of time, in seconds, after a scale in activity completes before another scale in activity can start.
        "ScaleOutCooldown": 300,  # ScaleOutCooldown - The amount of time, in seconds, after a scale out activity completes before another scale out activity can start.
        # 'DisableScaleIn': True|False - ndicates whether scale in by the target tracking policy is disabled.
        # If the value is true , scale in is disabled and the target tracking policy won't remove capacity from the scalable resource.
    },
)

The endpoint is now ready for invocation.

--- 
## 2. Using the Endpoint <a id='endpoint'></a>

### 2.1 Uploading the Request Payload <a id='upload'></a>

First, you need to upload the request to Amazon S3. We define a function called, `upload_file`, to make it easier to make multiple invocations in a later step.

In [12]:
import os


def upload_file(input_location):
    prefix = f"{bucket_prefix}/input"
    return sm_session.upload_data(
        input_location,
        bucket=sm_session.default_bucket(),
        key_prefix=prefix,
        extra_args={"ContentType": "text/libsvm"},
    )

### 2.1 Get Prediction from the Endpoint   <a id='invoke-endpoint'></a>

Get inferences from the model hosted on your asynchronous endpoint with the [`AsyncPredictor`](https://sagemaker.readthedocs.io/en/stable/api/inference/predictor_async.html#asyncpredictor)'s [`predict_async`](https://sagemaker.readthedocs.io/en/stable/api/inference/predictor_async.html#sagemaker.predictor_async.AsyncPredictor.predict_async) API. `input_path` specifies the location of your inference data.

In [13]:
input_1_location = "input/test_point_0.libsvm"
input_1_s3_location = upload_file(input_1_location)

In [14]:
# V3: invoke the async endpoint via core Endpoint.invoke_async.
# The response carries output_location (V2 called it output_path).
async_response = async_endpoint.invoke_async(
    input_location=input_1_s3_location,
    content_type="text/libsvm",
)
output_location = async_response.output_location

### 2.2 Check Output Location <a id='check-output'></a>

Check the output location to see if the inference has been processed. We make multiple requests (beginning of the `while True` statement in the `get_output` function) every two seconds until there is an output of the inference request: 

In [15]:
import urllib, time
from botocore.exceptions import ClientError


def get_output(output_location):
    output_url = urllib.parse.urlparse(output_location)
    bucket = output_url.netloc
    key = output_url.path[1:]
    while True:
        try:
            return sm_session.read_s3_file(bucket=output_url.netloc, key_prefix=output_url.path[1:])
        except ClientError as e:
            if e.response["Error"]["Code"] == "NoSuchKey":
                print("waiting for output...")
                time.sleep(2)
                continue
            raise

In [16]:
output = get_output(output_location)
print(f"Output: {output}")

waiting for output...


Output: 0.809421181678772


### 2.3 Multiple Invocations  <a id='multiple-invoke'></a>

The following shows how you can invoke the endpoint with multiple requests:

In [17]:
inferences = []
for i in range(25):
    input_file = f"input/test_point_{i}.libsvm"
    input_file_s3_location = upload_file(input_file)
    print(f"Invoking Endpoint with {input_file}")
    async_response = async_endpoint.invoke_async(
        input_location=input_file_s3_location,
        content_type="text/libsvm",
    )
    output_location = async_response.output_location
    print(output_location)
    inferences += [(input_file, output_location)]
    time.sleep(0.5)

for input_file, output_location in inferences:
    output = get_output(output_location)
    print(f"Input File: {input_file}, Output: {output}")

Invoking Endpoint with input/test_point_0.libsvm
s3://sagemaker-us-west-1-729646638167/DEMO-async-inference/output/d49ca133-c060-4171-aa38-b35dee122cef.out


Invoking Endpoint with input/test_point_1.libsvm
s3://sagemaker-us-west-1-729646638167/DEMO-async-inference/output/9f57c4c3-ac82-43e9-aef7-be9ab5f82859.out


Invoking Endpoint with input/test_point_2.libsvm
s3://sagemaker-us-west-1-729646638167/DEMO-async-inference/output/f48070f1-8365-47af-8a99-e55d6bb8f2cd.out


Invoking Endpoint with input/test_point_3.libsvm
s3://sagemaker-us-west-1-729646638167/DEMO-async-inference/output/d43b0d11-9ec8-4974-a42e-4ab043d7050e.out


Invoking Endpoint with input/test_point_4.libsvm
s3://sagemaker-us-west-1-729646638167/DEMO-async-inference/output/b21848c7-6468-45ca-a638-9bc7d0191b0c.out


...

Input File: input/test_point_10.libsvm, Output: 0.795388400554657
Input File: input/test_point_11.libsvm, Output: 0.3886236250400543
Input File: input/test_point_12.libsvm, Output: 0.9560881853103638
Input File: input/test_point_13.libsvm, Output: 0.3886236250400543


Input File: input/test_point_14.libsvm, Output: 0.5001285076141357
Input File: input/test_point_15.libsvm, Output: 0.45026054978370667
Input File: input/test_point_16.libsvm, Output: 0.8613329529762268


Input File: input/test_point_17.libsvm, Output: 0.5521252751350403
Input File: input/test_point_18.libsvm, Output: 0.6051725149154663
Input File: input/test_point_19.libsvm, Output: 0.5196762681007385


Input File: input/test_point_20.libsvm, Output: 0.3886236250400543
Input File: input/test_point_21.libsvm, Output: 0.42019087076187134
Input File: input/test_point_22.libsvm, Output: 0.4293918311595917
Input File: input/test_point_23.libsvm, Output: 0.3053364157676697


Input File: input/test_point_24.libsvm, Output: 0.9731993079185486


### 3. Clean up <a id='clean'></a>

If you enabled auto-scaling for your endpoint, ensure you deregister the endpoint as a scalable target before deleting the endpoint. To do this, run the following:

In [18]:
response = client.deregister_scalable_target(
    ServiceNamespace="sagemaker",
    ResourceId=resource_id,
    ScalableDimension="sagemaker:variant:DesiredInstanceCount",
)

Remember to delete your endpoint after use as you will be charged for the instances used in this Demo. 

In [19]:
# V3: delete the core resources explicitly (Endpoint.delete does not cascade).
from sagemaker.core.resources import Endpoint, EndpointConfig, Model

async_endpoint.delete()
try:
    EndpointConfig.get(endpoint_name).delete()
except Exception as e:
    print(f"EndpointConfig cleanup skipped: {e}")
try:
    Model.get(model_name).delete()
except Exception as e:
    print(f"Model cleanup skipped: {e}")

[07/16/26 10:04:02] INFO     Deleting Endpoint - AsyncInferenceDemo-Endpoint-2026-07-16-10-00-36 ]8;id=12565573;file:///Users/lucasjia/pysdk/pysdk-workspace/sagemaker-python-sdk-lucas/sagemaker-core/src/sagemaker/core/resources.py\resources.py]8;;\:]8;id=12565574;file:///Users/lucasjia/pysdk/pysdk-workspace/sagemaker-python-sdk-lucas/sagemaker-core/src/sagemaker/core/resources.py#10428\10428]8;;\

                    INFO     Deleting EndpointConfig -                                           ]8;id=12565580;file:///Users/lucasjia/pysdk/pysdk-workspace/sagemaker-python-sdk-lucas/sagemaker-core/src/sagemaker/core/resources.py\resources.py]8;;\:]8;id=12565581;file:///Users/lucasjia/pysdk/pysdk-workspace/sagemaker-python-sdk-lucas/sagemaker-core/src/sagemaker/core/resources.py#11220\11220]8;;\
                             AsyncInferenceDemo-Endpoint-2026-07-16-10-00-36                                       

                    INFO     Deleting Model - AsyncInferenceDemo-Model-2026-07-16-10-00-36       ]8;id=12565587;file:///Users/lucasjia/pysdk/pysdk-workspace/sagemaker-python-sdk-lucas/sagemaker-core/src/sagemaker/core/resources.py\resources.py]8;;\:]8;id=12565588;file:///Users/lucasjia/pysdk/pysdk-workspace/sagemaker-python-sdk-lucas/sagemaker-core/src/sagemaker/core/resources.py#20740\20740]8;;\

You may also want to delete any other resources you might have created such as SNS topics, S3 objects, etc.

## Notebook CI Test Results

This notebook was tested in multiple regions. The test results are as follows, except for us-west-2 which is shown at the top of the notebook.

![This us-east-1 badge failed to load. Check your device's internet connectivity, otherwise the service is currently unavailable](https://prod.us-west-2.tcx-beacon.docs.aws.dev/sagemaker-nb/us-east-1/deploy_and_monitor|sm-async_inference_with_python_sdk|sm-async_inference_with_python_sdk.ipynb)

![This us-east-2 badge failed to load. Check your device's internet connectivity, otherwise the service is currently unavailable](https://prod.us-west-2.tcx-beacon.docs.aws.dev/sagemaker-nb/us-east-2/deploy_and_monitor|sm-async_inference_with_python_sdk|sm-async_inference_with_python_sdk.ipynb)

![This us-west-1 badge failed to load. Check your device's internet connectivity, otherwise the service is currently unavailable](https://prod.us-west-2.tcx-beacon.docs.aws.dev/sagemaker-nb/us-west-1/deploy_and_monitor|sm-async_inference_with_python_sdk|sm-async_inference_with_python_sdk.ipynb)

![This ca-central-1 badge failed to load. Check your device's internet connectivity, otherwise the service is currently unavailable](https://prod.us-west-2.tcx-beacon.docs.aws.dev/sagemaker-nb/ca-central-1/deploy_and_monitor|sm-async_inference_with_python_sdk|sm-async_inference_with_python_sdk.ipynb)

![This sa-east-1 badge failed to load. Check your device's internet connectivity, otherwise the service is currently unavailable](https://prod.us-west-2.tcx-beacon.docs.aws.dev/sagemaker-nb/sa-east-1/deploy_and_monitor|sm-async_inference_with_python_sdk|sm-async_inference_with_python_sdk.ipynb)

![This eu-west-1 badge failed to load. Check your device's internet connectivity, otherwise the service is currently unavailable](https://prod.us-west-2.tcx-beacon.docs.aws.dev/sagemaker-nb/eu-west-1/deploy_and_monitor|sm-async_inference_with_python_sdk|sm-async_inference_with_python_sdk.ipynb)

![This eu-west-2 badge failed to load. Check your device's internet connectivity, otherwise the service is currently unavailable](https://prod.us-west-2.tcx-beacon.docs.aws.dev/sagemaker-nb/eu-west-2/deploy_and_monitor|sm-async_inference_with_python_sdk|sm-async_inference_with_python_sdk.ipynb)

![This eu-west-3 badge failed to load. Check your device's internet connectivity, otherwise the service is currently unavailable](https://prod.us-west-2.tcx-beacon.docs.aws.dev/sagemaker-nb/eu-west-3/deploy_and_monitor|sm-async_inference_with_python_sdk|sm-async_inference_with_python_sdk.ipynb)

![This eu-central-1 badge failed to load. Check your device's internet connectivity, otherwise the service is currently unavailable](https://prod.us-west-2.tcx-beacon.docs.aws.dev/sagemaker-nb/eu-central-1/deploy_and_monitor|sm-async_inference_with_python_sdk|sm-async_inference_with_python_sdk.ipynb)

![This eu-north-1 badge failed to load. Check your device's internet connectivity, otherwise the service is currently unavailable](https://prod.us-west-2.tcx-beacon.docs.aws.dev/sagemaker-nb/eu-north-1/deploy_and_monitor|sm-async_inference_with_python_sdk|sm-async_inference_with_python_sdk.ipynb)

![This ap-southeast-1 badge failed to load. Check your device's internet connectivity, otherwise the service is currently unavailable](https://prod.us-west-2.tcx-beacon.docs.aws.dev/sagemaker-nb/ap-southeast-1/deploy_and_monitor|sm-async_inference_with_python_sdk|sm-async_inference_with_python_sdk.ipynb)

![This ap-southeast-2 badge failed to load. Check your device's internet connectivity, otherwise the service is currently unavailable](https://prod.us-west-2.tcx-beacon.docs.aws.dev/sagemaker-nb/ap-southeast-2/deploy_and_monitor|sm-async_inference_with_python_sdk|sm-async_inference_with_python_sdk.ipynb)

![This ap-northeast-1 badge failed to load. Check your device's internet connectivity, otherwise the service is currently unavailable](https://prod.us-west-2.tcx-beacon.docs.aws.dev/sagemaker-nb/ap-northeast-1/deploy_and_monitor|sm-async_inference_with_python_sdk|sm-async_inference_with_python_sdk.ipynb)

![This ap-northeast-2 badge failed to load. Check your device's internet connectivity, otherwise the service is currently unavailable](https://prod.us-west-2.tcx-beacon.docs.aws.dev/sagemaker-nb/ap-northeast-2/deploy_and_monitor|sm-async_inference_with_python_sdk|sm-async_inference_with_python_sdk.ipynb)

![This ap-south-1 badge failed to load. Check your device's internet connectivity, otherwise the service is currently unavailable](https://prod.us-west-2.tcx-beacon.docs.aws.dev/sagemaker-nb/ap-south-1/deploy_and_monitor|sm-async_inference_with_python_sdk|sm-async_inference_with_python_sdk.ipynb)
